# 📊 พยากรณ์แนวโน้มการขึ้นทะเบียนเกษตรกรและพื้นที่เกษตรกรรมรายอำเภอ (Agricultural Registration Forecasting)
## คลาสเรียนรู้ Machine Learning สำหรับพยากรณ์ข้อมูลอนุกรมเวลา (Time-Series) เพื่อประเมินความเสี่ยงและแผนงบประมาณช่วยเหลือภัยพิบัติ
### 📊 แสดงผลแผนภูมิด้วย Plotly (Interactive Chart) เพื่อความสวยงามระดับ Premium และรองรับภาษาไทย 100%

---

### 📋 วัตถุประสงค์
1. วิเคราะห์และสำรวจแนวโน้มพื้นที่การเกษตร (`เนื้อที่(ไร่)`) และจำนวนครัวเรือนเกษตรกรรายอำเภอในจังหวัดเพชรบูรณ์ (ปี 2561 - 2568)
2. เรียนรู้วิธีการทำพยากรณ์อนุกรมเวลา (Time-Series Forecasting) บนข้อมูลขนาดเล็กโดยใช้ฟีเจอร์แนวโน้มเวลา (Time Trend) และค่าหน่วงเวลา (Lag Features)
3. ฝึกสอนโมเดล **Linear Regression** และ **Gradient Boosting Regressor** บนข้อมูลการขึ้นทะเบียน
4. ทำนายอนาคต 2 ปีข้างหน้า (ปี 2569 - 2570) แบบวนซ้ำ (Recursive Forecasting) เพื่อประเมินแนวโน้มพื้นที่การเกษตรที่ต้องเตรียมแผนงบประมาณรองรับภัยพิบัติ

## 1. Setup & Import Libraries 📦

In [1]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# ตั้งค่า renderer สำหรับ VS Code Jupyter Notebook ให้แสดงผลกราฟิกแบบตอบสนองได้
pio.renderers.default = 'notebook_connected'
pio.templates.default = 'plotly_white'

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print("✅ Setup สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)")

✅ Setup สำเร็จ และพร้อมใช้งาน (ใช้ Plotly สำหรับการวาดกราฟ)


## 2. Load and Clean Dataset 📁
ทำการโหลดข้อมูลการขึ้นทะเบียนเกษตรกรในจังหวัดเพชรบูรณ์

In [2]:
# กำหนด path ของข้อมูลการขึ้นทะเบียนเกษตรกร
file_path = os.path.join("data", "การขึ้นทะเบียนเกษตรกร", "การขึ้นทะเบียนเกษตรกร.csv")

# โหลดข้อมูล
df = pd.read_csv(file_path, encoding='utf-8-sig')

# ทำความสะอาดคอลัมน์และข้อมูลข้อความ
df.columns = [col.strip() for col in df.columns]
df['อำเภอ'] = df['อำเภอ'].str.strip()

print(f"📊 โหลดข้อมูลการขึ้นทะเบียนสำเร็จ! จำนวนแถวทั้งหมด: {len(df)} แถว")
print("รายชื่ออำเภอที่พบ:", df['อำเภอ'].unique())
display(df.head(11))

📊 โหลดข้อมูลการขึ้นทะเบียนสำเร็จ! จำนวนแถวทั้งหมด: 88 แถว
รายชื่ออำเภอที่พบ: <StringArray>
['เมืองเพชรบูรณ์',          'ชนแดน',        'หล่มสัก',       'หล่มเก่า',
    'วิเชียรบุรี',         'ศรีเทพ',        'หนองไผ่',      'บึงสามพัน',
        'น้ำหนาว',        'วังโป่ง',         'เขาค้อ']
Length: 11, dtype: str


,ปี,อำเภอ,จำนวนครัวเรือน,จำนวนแปลง,เนื้อที่(ไร่)
0,2561,เมืองเพชรบูรณ์,15248,44470,314322.39
1,2561,ชนแดน,8743,19170,282037.54
2,2561,หล่มสัก,14560,39290,210108.37
3,2561,หล่มเก่า,8410,23317,213199.92
4,2561,วิเชียรบุรี,8509,19068,219649.00
5,2561,ศรีเทพ,7737,17837,202869.88
6,2561,หนองไผ่,11686,36187,410938.67
7,2561,บึงสามพัน,5170,11734,163691.29
8,2561,น้ำหนาว,2188,7971,100572.53
9,2561,วังโป่ง,3518,9566,120613.81


## 3. Exploratory Data Analysis (EDA) 📈
พล็อตประวัติแนวโน้มความเปลี่ยนแปลงของพื้นที่การเกษตรรายอำเภอ

In [3]:
# พล็อตข้อมูลดิบของเนื้อที่เกษตรกรรมที่ลงทะเบียนแยกตามอำเภอ
fig = px.line(
    df,
    x='ปี',
    y='เนื้อที่(ไร่)',
    color='อำเภอ',
    markers=True,
    title='แนวโน้มพื้นที่การเกษตรที่ขึ้นทะเบียนรายอำเภอในจังหวัดเพชรบูรณ์ (ปี 2561 - 2568)'
)
fig.update_layout(
    xaxis_title='ปีงบประมาณ',
    yaxis_title='พื้นที่การเกษตรที่ขึ้นทะเบียน (ไร่)',
    title_x=0.5,
    height=550
)
fig.show()

## 4. Feature Engineering & Time-Series Preparation 🛠️
สร้างตัวแปรต้น (Features) ได้แก่:
- `year_idx`: ดัชนีปีงบประมาณเชิงเส้น (ปี - 2561) เพื่อเป็นตัวแปรบอกแนวโน้มเวลา
- `lag_1_area`: พื้นที่เกษตรของปีงบประมาณที่แล้ว
- `lag_1_households`: จำนวนครัวเรือนของปีงบประมาณที่แล้ว
- `district_encoded`: ตัวแปรจัดกลุ่มอำเภอ (One-Hot Encoding)

In [4]:
# เรียงลำดับข้อมูลตามอำเภอและปี เพื่อสร้าง Lag Features
df_prep = df.sort_values(by=['อำเภอ', 'ปี']).reset_index(drop=True)

# สร้าง Lag 1 ปี
df_prep['lag_1_area'] = df_prep.groupby('อำเภอ')['เนื้อที่(ไร่)'].shift(1)
df_prep['lag_1_households'] = df_prep.groupby('อำเภอ')['จำนวนครัวเรือน'].shift(1)
df_prep['lag_1_plots'] = df_prep.groupby('อำเภอ')['จำนวนแปลง'].shift(1)

# สร้างดัชนีปี
df_prep['year_idx'] = df_prep['ปี'] - 2561

# ลบแถวปีแรก (2561) ที่ไม่มี Lag ย้อนหลัง (เป็น NaN)
df_model = df_prep.dropna().reset_index(drop=True)

# ทำ One-Hot Encoding สำหรับตัวแปรกองอำเภอ
df_model = pd.get_dummies(df_model, columns=['อำเภอ'], prefix='dist', drop_first=False)

print("📊 โครงสร้างข้อมูลสำหรับฝึกสอนโมเดล (Features & Target):")
print(df_model.info())
display(df_model.head(5))

📊 โครงสร้างข้อมูลสำหรับฝึกสอนโมเดล (Features & Target):
<class 'pandas.DataFrame'>
RangeIndex: 77 entries, 0 to 76
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ปี                   77 non-null     int64  
 1   จำนวนครัวเรือน       77 non-null     int64  
 2   จำนวนแปลง            77 non-null     int64  
 3   เนื้อที่(ไร่)        77 non-null     float64
 4   lag_1_area           77 non-null     float64
 5   lag_1_households     77 non-null     float64
 6   lag_1_plots          77 non-null     float64
 7   year_idx             77 non-null     int64  
 8   dist_ชนแดน           77 non-null     bool   
 9   dist_น้ำหนาว         77 non-null     bool   
 10  dist_บึงสามพัน       77 non-null     bool   
 11  dist_วังโป่ง         77 non-null     bool   
 12  dist_วิเชียรบุรี     77 non-null     bool   
 13  dist_ศรีเทพ          77 non-null     bool   
 14  dist_หนองไผ่         77 non-null     bool   
 1

,ปี,จำนวนครัวเรือน,จำนวนแปลง,เนื้อที่(ไร่),lag_1_area,lag_1_households,lag_1_plots,year_idx,dist_ชนแดน,dist_น้ำหนาว,dist_บึงสามพัน,dist_วังโป่ง,dist_วิเชียรบุรี,dist_ศรีเทพ,dist_หนองไผ่,dist_หล่มสัก,dist_หล่มเก่า,dist_เขาค้อ,dist_เมืองเพชรบูรณ์
0,2562,10115,25226,315201.31,282037.54,8743.0,19170.0,1,True,False,False,False,False,False,False,False,False,False,False
1,2563,11208,30602,440556.20,315201.31,10115.0,25226.0,2,True,False,False,False,False,False,False,False,False,False,False
2,2564,10388,27579,387814.11,440556.20,11208.0,30602.0,3,True,False,False,False,False,False,False,False,False,False,False
3,2565,9391,24198,336926.60,387814.11,10388.0,27579.0,4,True,False,False,False,False,False,False,False,False,False,False
4,2566,9181,31086,430987.65,336926.60,9391.0,24198.0,5,True,False,False,False,False,False,False,False,False,False,False


## 5. Train-Test Split & Modeling 🤖
สำหรับโมเดลอนุกรมเวลา เราแบ่งกลุ่มทดสอบ (Test Set) ตามปีงบประมาณ:
- **Train Set**: ข้อมูลปี 2562 ถึง 2567
- **Test Set**: ข้อมูลปีล่าสุด 2568 (ใช้ประเมินความแม่นยำ)

In [5]:
# คัดเลือก Features และ Target
dist_cols = [col for col in df_model.columns if col.startswith('dist_')]
features = ['year_idx', 'lag_1_area', 'lag_1_households', 'lag_1_plots'] + dist_cols
target_area = 'เนื้อที่(ไร่)'
target_households = 'จำนวนครัวเรือน'
target_plots = 'จำนวนแปลง'

# แบ่งกลุ่มข้อมูล
train_mask = df_model['ปี'] < 2568
test_mask = df_model['ปี'] == 2568

df_train = df_model[train_mask]
df_test = df_model[test_mask]

X_train, y_train_area = df_train[features], df_train[target_area]
X_test, y_test_area = df_test[features], df_test[target_area]

y_train_hh, y_test_hh = df_train[target_households], df_test[target_households]
y_train_plots, y_test_plots = df_train[target_plots], df_test[target_plots]

print(f"📂 ข้อมูล Train (ปี 2562-2567): {len(df_train)} แถว")
print(f"📂 ข้อมูล Test (ปี 2568): {len(df_test)} แถว")

# 1. ฝึกสอนโมเดลสำหรับทำนายพื้นที่การเกษตร (Area Forecast)
lr_area = LinearRegression().fit(X_train, y_train_area)
gbr_area = GradientBoostingRegressor(loss='huber', random_state=42).fit(X_train, y_train_area)

# 2. ฝึกสอนโมเดลสำหรับทำนายจำนวนครัวเรือน (Households Forecast)
lr_hh = LinearRegression().fit(X_train, y_train_hh)
gbr_hh = GradientBoostingRegressor(loss='huber', random_state=42).fit(X_train, y_train_hh)

# 3. ฝึกสอนโมเดลสำหรับทำนายจำนวนแปลง (Plots Forecast)
lr_plots = LinearRegression().fit(X_train, y_train_plots)
gbr_plots = GradientBoostingRegressor(loss='huber', random_state=42).fit(X_train, y_train_plots)

# ทดสอบประเมินประสิทธิภาพสำหรับเป้าหมายพื้นที่การเกษตรในปี 2568
lr_pred_area = lr_area.predict(X_test)
gbr_pred_area = gbr_area.predict(X_test)

print(f"\n{'='*20} 🎯 ประสิทธิภาพการทำนายพื้นที่ปี 2568 {'='*20}")
print(f"[Linear Regression] R2: {r2_score(y_test_area, lr_pred_area):.4f} | MAE: {mean_absolute_error(y_test_area, lr_pred_area):,.2f}")
print(f"[Gradient Boosting] R2: {r2_score(y_test_area, gbr_pred_area):.4f} | MAE: {mean_absolute_error(y_test_area, gbr_pred_area):,.2f}")

📂 ข้อมูล Train (ปี 2562-2567): 66 แถว
📂 ข้อมูล Test (ปี 2568): 11 แถว

==================== 🎯 ประสิทธิภาพการทำนายพื้นที่ปี 2568 ====================
[Linear Regression] R2: 0.7337 | MAE: 41,113.48
[Gradient Boosting] R2: 0.6688 | MAE: 40,134.68


## 6. Recursive Future Forecasting (ปี 2569 - 2570) 🔮
เราจะทำนายอนาคตโดยใช้การทำนายแบบวนซ้ำ (Recursive Forecasting):
1. นำผลลัพธ์ของปี 2568 ไปเป็นค่า Lag 1 เพื่อทำนายปี 2569
2. นำผลลัพธ์ที่ได้ของปี 2569 ไปเป็นค่า Lag 1 เพื่อทำนายปี 2570

In [6]:
# รายชื่ออำเภอทั้งหมด
districts = df['อำเภอ'].unique()

# เก็บผลลัพธ์ประวัติการทำนายรายอำเภอ
forecast_results = []

for dist in districts:
    # ดึงค่าจริงล่าสุดของปี 2568
    dist_latest = df_prep[(df_prep['อำเภอ'] == dist) & (df_prep['ปี'] == 2568)].iloc[0]
    
    current_area = dist_latest['เนื้อที่(ไร่)']
    current_hh = dist_latest['จำนวนครัวเรือน']
    current_plots = dist_latest['จำนวนแปลง']
    
    # เตรียมเวกเตอร์ของ One-Hot สำหรับอำเภอนี้
    dist_onehot = {col: 1 if col == f'dist_{dist}' else 0 for col in dist_cols}
    
    # ทำนายแบบวนซ้ำสำหรับปี 2569 และ 2570
    for year in [2569, 2570]:
        year_idx = year - 2561
        
        # เตรียมเวกเตอร์สำหรับทำนาย
        input_data = {
            'year_idx': year_idx,
            'lag_1_area': current_area,
            'lag_1_households': current_hh,
            'lag_1_plots': current_plots
        }
        input_data.update(dist_onehot)
        
        # สร้าง DataFrame 1 แถวสำหรับใส่เข้าโมเดล
        X_input = pd.DataFrame([input_data])[features]
        
        # ทำนายด้วย Gradient Boosting (โมเดลที่ดีที่สุดและทนทานต่อ outliers)
        pred_area = gbr_area.predict(X_input)[0]
        pred_hh = gbr_hh.predict(X_input)[0]
        pred_plots = gbr_plots.predict(X_input)[0]
        
        # บันทึกผลลัพธ์
        forecast_results.append({
            'ปี': year,
            'อำเภอ': dist,
            'เนื้อที่(ไร่)': max(0, pred_area),
            'จำนวนครัวเรือน': max(0, int(pred_hh)),
            'จำนวนแปลง': max(0, int(pred_plots)),
            'ประเภท': 'พยากรณ์ (Forecast)'
        })
        
        # อัปเดตค่าสำหรับวนลูปทำนายปีถัดไป
        current_area = pred_area
        current_hh = pred_hh
        current_plots = pred_plots

df_forecast = pd.DataFrame(forecast_results)
print("🔮 ผลลัพธ์การทำนายอนาคต (ตัวอย่าง 10 แถวแรก):")
display(df_forecast.head(10))

🔮 ผลลัพธ์การทำนายอนาคต (ตัวอย่าง 10 แถวแรก):


,ปี,อำเภอ,เนื้อที่(ไร่),จำนวนครัวเรือน,จำนวนแปลง,ประเภท
0,2569,เมืองเพชรบูรณ์,316936.280504,16750,48624,พยากรณ์ (Forecast)
1,2570,เมืองเพชรบูรณ์,321866.166441,16671,48624,พยากรณ์ (Forecast)
2,2569,ชนแดน,260833.400463,8825,21799,พยากรณ์ (Forecast)
3,2570,ชนแดน,235046.655740,8655,21133,พยากรณ์ (Forecast)
4,2569,หล่มสัก,222276.118251,14154,33887,พยากรณ์ (Forecast)
5,2570,หล่มสัก,227411.708487,13854,36423,พยากรณ์ (Forecast)
6,2569,หล่มเก่า,189082.543405,8988,26581,พยากรณ์ (Forecast)
7,2570,หล่มเก่า,189082.543405,9139,25908,พยากรณ์ (Forecast)
8,2569,วิเชียรบุรี,430053.590751,12633,35898,พยากรณ์ (Forecast)
9,2570,วิเชียรบุรี,438139.958706,12624,33646,พยากรณ์ (Forecast)


## 7. Visualization 📊
พล็อตเปรียบเทียบแนวโน้มข้อมูลในอดีต (2561-2568) และคำทำนายความต้องการรองรับภัยพิบัติในอนาคต (2569-2570)

In [7]:
# สร้าง DataFrame รวมระหว่างอดีตจริงและอนาคตพยากรณ์
df_historical = df[['ปี', 'อำเภอ', 'เนื้อที่(ไร่)', 'จำนวนครัวเรือน', 'จำนวนแปลง']].copy()
df_historical['ประเภท'] = 'ข้อมูลจริง (Actual)'

df_combined = pd.concat([df_historical, df_forecast], ignore_index=True)

# แสดงเส้นกราฟพยากรณ์พื้นที่การเกษตรรายอำเภอ
fig = px.line(
    df_combined,
    x='ปี',
    y='เนื้อที่(ไร่)',
    color='อำเภอ',
    line_dash='ประเภท',
    markers=True,
    title='พยากรณ์แนวโน้มเนื้อที่ทำการเกษตรรายอำเภอ จังหวัดเพชรบูรณ์ (ปี 2561 - 2570)'
)
fig.update_layout(
    xaxis_title='ปีงบประมาณ',
    yaxis_title='พื้นที่ลงทะเบียน (ไร่)',
    title_x=0.5,
    height=600,
    hovermode='x unified'
)
fig.show()

### 7.2 พยากรณ์ครัวเรือนที่ขึ้นทะเบียน (Farmers Households Forecast)
แสดงแนวโน้มจำนวนครัวเรือนผู้ประกอบการเพื่อเตรียมแผนงบประมาณความรับผิดชอบเชิงพื้นที่

In [8]:
fig_hh = px.line(
    df_combined,
    x='ปี',
    y='จำนวนครัวเรือน',
    color='อำเภอ',
    line_dash='ประเภท',
    markers=True,
    title='พยากรณ์แนวโน้มจำนวนครัวเรือนเกษตรกรรายอำเภอ จังหวัดเพชรบูรณ์ (ปี 2561 - 2570)'
)
fig_hh.update_layout(
    xaxis_title='ปีงบประมาณ',
    yaxis_title='จำนวนครัวเรือน (ครัวเรือน)',
    title_x=0.5,
    height=600,
    hovermode='x unified'
)
fig_hh.show()

## 8. Feature Importance of GBR Models 🏅
วิเคราะห์ตัวแปรต้นที่มีอิทธิพลต่อผลลัพธ์โมเดลพยากรณ์ความต้องการจัดสรรความช่วยเหลือมากที่สุด

In [9]:
importances = gbr_area.feature_importances_
indices = np.argsort(importances)[::-1]

imp_df = pd.DataFrame({
    'Feature': [features[i] for i in indices],
    'Importance': importances[indices]
}).head(8)

# แสดงแผนภูมิแท่งความสำคัญของตัวแปร โดยใช้โทนสีม่วงเข้มอย่างชัดเจน
fig_imp = px.bar(
    imp_df.sort_values('Importance', ascending=True),
    y='Feature',
    x='Importance',
    orientation='h',
    title='ปัจจัยที่มีอิทธิพลต่อการพยากรณ์พื้นที่ทำการเกษตรรายจังหวัด',
    color='Importance',
    color_continuous_scale=['#9575CD', '#311B92'] # สีม่วงคมชัดตัดขอบสวยงาม
)
fig_imp.update_layout(
    yaxis_title='ปัจจัย/ตัวแปรต้น',
    xaxis_title='ค่าน้ำหนักความสำคัญ (Relative Importance)',
    coloraxis_showscale=False,
    height=400,
    title_x=0.5
)
fig_imp.show()